In [1]:
'''
Group 38 — Evaluation & Inference notebook
APPROACH EXPLANATION
-------------------------------------------------------------------
Model architecture and roadmap

EfficientNet-B3, with the final classification head replaced by a Linear(1536 -> 2) layer -> binary output
    Input resolution: 384x384 (higher than the B3 default of 300x300 —
    better detail for subtle surface defects)

IMPORTANT key design decisions - 
   we use the provided COCO annotations (category_id == 22) to crop each image to the bottle body before passing it to the model ->
   this removes irrelevant background, letting the model focus on the surface

Inference strategy
---------------------------------------------------------------------------------
   Test Time Augmentation x4: The final prediction is an ensemble of probabilities across 4 rotations (0, 90, 180, 270 deg) of the same image
   This significantly reduces variance and boosts the model s robustness on the hidden test set
   No data augmentations (like flip or color jitter) are applied here to keep predictions deterministic!!!
'''

'\nGroup 38 — Evaluation & Inference notebook\nAPPROACH EXPLANATION\n-------------------------------------------------------------------\nModel architecture and roadmap\n\nEfficientNet-B3, with the final classification head replaced by a Linear(1536 -> 2) layer -> binary output\n    Input resolution: 384x384 (higher than the B3 default of 300x300 —\n    better detail for subtle surface defects)\n\nIMPORTANT key design decisions - \n   we use the provided COCO annotations (category_id == 22) to crop each image to the bottle body before passing it to the model ->\n   this removes irrelevant background, letting the model focus on the surface\n\nInference strategy\n---------------------------------------------------------------------------------\n   Test Time Augmentation x4: The final prediction is an ensemble of probabilities across 4 rotations (0, 90, 180, 270 deg) of the same image\n   This significantly reduces variance and boosts the model s robustness on the hidden test set\n   No d

In [2]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import torch
import torch.nn as nn
import os
import pandas as pd
import json

def roi_def(json_path):
    with open(json_path, 'r') as read_annotations:
        data = json.load(read_annotations)
        
    # Linking photo id and file name
    id_to_name = {img['id']: img['file_name'] for img in data['images']}
    roi_dict = {}
    
    for annot in data['annotations']:
        if annot['category_id'] == 22:
            img_name = id_to_name[annot['image_id']]
            roi_dict[img_name] = annot['bbox']      
    print(f"{len(roi_dict)} ROI to crop - found!")
    return roi_dict

In [3]:
# test transform -> eval notebook (only resize and normalize)
test_transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [4]:
class BottleTestDataset(Dataset):
    def __init__(self, img_dir, roi_dict):
        self.img_dir = img_dir
        self.img_names = os.listdir(img_dir)
        self.roi_dict = roi_dict

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_name = self.img_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        
        # ROI crop
        if img_name in self.roi_dict:
            x, y, width, height = self.roi_dict[img_name]
            image = image.crop((x, y, x + width, y + height)) # cropping based on ROI
            
        #returning filename alongside the image for the submission file
        return img_name, test_transform(image)

In [5]:
#config setup
TEST_DIR = '/kaggle/input/competitions/1st-krones-vision-ai-challenge/test_images'
TEST_JSON = '/kaggle/input/competitions/1st-krones-vision-ai-challenge/test_annotations_roi_only.json'
MODEL_SAVED_PATH = '/kaggle/input/notebooks/dreval/38-training-notebook/best_model.pth'
SUBMISSION_FILE = 'test_predictions.csv'
BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading test data") 
test_roi_dict = roi_def(TEST_JSON)
test_loader = DataLoader(BottleTestDataset(TEST_DIR, test_roi_dict), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

#init model
print("Init model - EFFICIENTNET-B3")
model = models.efficientnet_b3(weights=None) # weights=None because we load our own model
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 2)

Loading test data
4418 ROI to crop - found!
Init model - EFFICIENTNET-B3


In [6]:
#load trained weights
print("Loading saved model weights...")
model.load_state_dict(torch.load(MODEL_SAVED_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

#evaluation (TTA)
print("Starting Inference: TTA x4 (0, 90, 180, 270)")
results = []

with torch.no_grad():
    for names, imgs in test_loader:
        imgs = imgs.to(DEVICE)

        with torch.amp.autocast('cuda'):
            # Original
            probs_orig = torch.softmax(model(imgs), dim=1)
            # 90 degrees
            probs_90 = torch.softmax(model(torch.rot90(imgs, k=1, dims=[2, 3])), dim=1)
            #180 degrees
            probs_180 = torch.softmax(model(torch.rot90(imgs, k=2, dims=[2, 3])), dim=1)
            # 270 degrees
            probs_270 = torch.softmax(model(torch.rot90(imgs, k=3, dims=[2, 3])), dim=1)

            #Average probabilities
            outputs = (probs_orig + probs_90 + probs_180 + probs_270) / 4.0
            
        _, predictions = torch.max(outputs, 1)

        for name, pred in zip(names, predictions):
            results.append({
                "image_id": name,
                "target": pred.item()
            })

df_result = pd.DataFrame(results)
df_result.to_csv(SUBMISSION_FILE, index=False)
print(f"Done! Predictions saved to {SUBMISSION_FILE}")

Loading saved model weights...
Starting Inference: TTA x4 (0, 90, 180, 270)
Done! Predictions saved to test_predictions.csv
